In [1]:
import os 
os.chdir('..')

In [2]:
# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Train Test Split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Fast Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [18]:
# Cross Validation
from sklearn.model_selection import cross_val_score

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# Pipeline (clean workflow)
from sklearn.pipeline import Pipeline

In [39]:
df=pd.read_csv(r'RAW_DATA1\data3.csv')

In [40]:
df.head()

,city,date,wind_speed,temperature,relative_humidity,pressure,visibility,Dust_Storm,Dust_next_day,month,pressure_change,wind_lag1,pressure_lag1,humidity_lag1,temp_lag1,wind_lag2,pressure_lag2,humidity_lag2,temp_lag2,high_wind_flag
0,Ahmedabad,2010-01-03,6.7,26.0,50.463976,1015.0,2000.0,0,0.0,1,1.7,6.7,1013.3,26.788533,29.4,9.8,1012.2,22.835358,27.2,0
1,Ahmedabad,2010-01-04,6.2,26.0,34.166863,1014.0,2000.0,0,0.0,1,-1.0,6.7,1015.0,50.463976,26.0,6.7,1013.3,26.788533,29.4,0
2,Ahmedabad,2010-01-05,999.9,999.9,0.000117,1012.0,2000.0,0,0.0,1,-2.0,6.2,1014.0,34.166863,26.0,6.7,1015.0,50.463976,26.0,1
3,Ahmedabad,2010-01-06,6.2,26.6,17.965870,1011.4,2000.0,0,0.0,1,-0.6,999.9,1012.0,0.000117,999.9,6.2,1014.0,34.166863,26.0,0
4,Ahmedabad,2010-01-07,6.7,27.2,31.649219,1013.0,2000.0,0,0.0,1,1.6,6.2,1011.4,17.965870,26.6,999.9,1012.0,0.000117,999.9,0


In [42]:
df.columns

Index(['city', 'date', 'wind_speed', 'temperature', 'relative_humidity',
       'pressure', 'visibility', 'Dust_Storm', 'Dust_next_day', 'month',
       'pressure_change', 'wind_lag1', 'pressure_lag1', 'humidity_lag1',
       'temp_lag1', 'wind_lag2', 'pressure_lag2', 'humidity_lag2', 'temp_lag2',
       'high_wind_flag'],
      dtype='object')

In [43]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ====================================
# 1️⃣ Create Clean Copy
# ====================================

df1 = df.copy()

# Remove unwanted columns
df1 = df1.drop(["city", "date", "Dust_Storm"], axis=1)

# Convert target to int (safety)
df1["Dust_next_day"] = df1["Dust_next_day"].astype(int)

# ====================================
# 2️⃣ Handle 999.9 (Environmental Missing Value)
# ====================================

df1 = df1.replace(999.9, np.nan)
df1 = df1.dropna()

# ====================================
# 3️⃣ Outlier Handling (IQR Capping)
# ====================================

cols_to_treat = df1.drop(
    ["Dust_next_day", "high_wind_flag"],
    axis=1
).columns

for col in cols_to_treat:
    
    Q1 = df1[col].quantile(0.25)
    Q3 = df1[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df1[col] = np.where(df1[col] < lower, lower, df1[col])
    df1[col] = np.where(df1[col] > upper, upper, df1[col])

# ====================================
# 4️⃣ Define X and y
# ====================================

X = df1.drop("Dust_next_day", axis=1)
y = df1["Dust_next_day"]

# ====================================
# 5️⃣ Train Test Split
# ====================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (16243, 16)
Test Shape: (4061, 16)


In [44]:
X_train

,wind_speed,temperature,relative_humidity,pressure,visibility,month,pressure_change,wind_lag1,pressure_lag1,humidity_lag1,temp_lag1,wind_lag2,pressure_lag2,humidity_lag2,temp_lag2,high_wind_flag
21884,7.7,39.0,28.670052,1000.1,2000.0,6.0,1.5,6.7,998.6,40.086086,41.6,3.10,999.1,28.453370,43.0,0
10955,6.7,42.4,23.329280,994.5,2000.0,6.0,-0.3,7.2,994.8,24.606020,40.4,9.25,995.8,27.667058,39.0,0
3958,4.1,34.0,18.592703,1011.1,2000.0,11.0,-1.0,3.6,1012.1,17.579891,34.0,3.10,1012.4,15.294664,34.0,0
3983,4.1,32.0,30.379563,1013.2,2500.0,11.0,-1.9,5.1,1015.1,24.755218,29.0,6.70,1016.7,28.664942,29.2,0
1928,4.1,37.0,29.909869,1008.6,2750.0,4.0,1.3,3.1,1007.3,30.567508,36.0,4.60,1007.5,36.227374,34.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6893,1.0,31.8,20.172608,1007.4,2000.0,11.0,-0.6,1.0,1008.0,16.064659,31.0,1.00,1012.0,14.412613,29.0,0
7659,0.0,33.5,15.728276,1006.1,2750.0,5.0,2.6,1.0,1003.5,19.057166,39.0,1.00,1002.7,20.710777,39.6,0
17934,1.0,29.0,76.274196,996.9,2000.0,8.0,1.7,1.0,995.2,60.529644,34.8,1.00,997.6,69.788132,31.8,0
17311,2.1,24.4,46.506327,1013.8,750.0,11.0,1.8,3.1,1012.0,46.033933,26.0,2.10,1011.5,39.226841,26.6,0


In [45]:
y_train

21884    1
10955    0
3958     0
3983     0
1928     0
        ..
6893     0
7659     0
17934    0
17311    0
23925    1
Name: Dust_next_day, Length: 16243, dtype: int64

In [46]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(16243, 16)
(4061, 16)
(16243,)
(4061,)


In [48]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20304 entries, 0 to 24645
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   wind_speed         20304 non-null  float64
 1   temperature        20304 non-null  float64
 2   relative_humidity  20304 non-null  float64
 3   pressure           20304 non-null  float64
 4   visibility         20304 non-null  float64
 5   Dust_next_day      20304 non-null  int64  
 6   month              20304 non-null  float64
 7   pressure_change    20304 non-null  float64
 8   wind_lag1          20304 non-null  float64
 9   pressure_lag1      20304 non-null  float64
 10  humidity_lag1      20304 non-null  float64
 11  temp_lag1          20304 non-null  float64
 12  wind_lag2          20304 non-null  float64
 13  pressure_lag2      20304 non-null  float64
 14  humidity_lag2      20304 non-null  float64
 15  temp_lag2          20304 non-null  float64
 16  high_wind_flag     20304 no

In [58]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# ==============================
# 2️⃣ Model Dictionary
# ==============================

models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000),
    "Decision_Tree": DecisionTreeClassifier(),
    "Random_Forest": RandomForestClassifier(n_estimators=100),
    "Naive_Bayes": GaussianNB(),
    "Ridge_Classifier": RidgeClassifier(),
    "SGD_Classifier": SGDClassifier()
}

results = {}

# ==============================
# 4️⃣ Training Loop with SMOTE
# ==============================

for name, model in models.items():
    
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Predict
    y_pred = pipe.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    cv_f1 = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    ).mean()
    
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        "Model_Object": pipe,
        "Accuracy": acc,
        "Recall": recall,
        "F1_Score": f1,
        "CV_F1_Score": cv_f1,
        "Confusion_Matrix": cm
    }

# ==============================
# 5️⃣ Compare Results
# ==============================

results_df = pd.DataFrame(results).T

results_df = results_df[[
    "Accuracy",
    "Recall",
    "F1_Score",
    "CV_F1_Score"
]]

results_df.sort_values("F1_Score", ascending=False)

,Accuracy,Recall,F1_Score,CV_F1_Score
Random_Forest,0.858656,0.441121,0.451243,0.447119
Logistic_Regression,0.728638,0.770093,0.42783,0.430127
SGD_Classifier,0.742428,0.723364,0.425275,0.422329
Ridge_Classifier,0.723713,0.775701,0.425205,0.426913
Naive_Bayes,0.71214,0.8,0.422716,0.409631
Decision_Tree,0.797094,0.368224,0.323481,0.349148


Why F1 Low?

Because:

Class imbalance heavy

Weather data noisy hota hai

Dust storm complex phenomenon hai

Linear boundaries struggle karte hain

In [59]:
best_model_name = results_df["F1_Score"].idxmax()
best_model = results[best_model_name]["Model_Object"]

print("Best Model Based on F1:", best_model_name)

Best Model Based on F1: Random_Forest


In [60]:
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

# Pipeline
pipe_nb = Pipeline([
    ("scaler", StandardScaler()),
    ("nb", GaussianNB())
])

# Hyperparameter grid
param_grid = {
    "nb__var_smoothing": np.logspace(-12, -6, 50)
}

grid_nb = GridSearchCV(
    pipe_nb,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_nb.fit(X_train, y_train)

print("Best Parameters:", grid_nb.best_params_)
print("Best CV F1:", grid_nb.best_score_)

Best Parameters: {'nb__var_smoothing': np.float64(1e-12)}
Best CV F1: 0.44432144425796044


In [61]:
best_nb = grid_nb.best_estimator_

y_pred_nb = best_nb.predict(X_test)

from sklearn.metrics import f1_score, recall_score, classification_report

print("Test F1:", f1_score(y_test, y_pred_nb))
print("Test Recall:", recall_score(y_test, y_pred_nb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))

Test F1: 0.4483173076923077
Test Recall: 0.697196261682243

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.79      0.86      3526
           1       0.33      0.70      0.45       535

    accuracy                           0.77      4061
   macro avg       0.64      0.74      0.65      4061
weighted avg       0.86      0.77      0.80      4061



In [62]:
import numpy as np
from sklearn.metrics import f1_score

probs = best_nb.predict_proba(X_test)[:,1]

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1 = 0
best_threshold = 0.5

for t in thresholds:
    y_pred_t = (probs >= t).astype(int)
    score = f1_score(y_test, y_pred_t)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best Threshold:", best_threshold)
print("Best F1 After Threshold Tuning:", best_f1)

Best Threshold: 0.8500000000000002
Best F1 After Threshold Tuning: 0.4565381708238851


In [63]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Pipeline
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(random_state=42))
])

# Hyperparameter Grid
param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 20],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid,
    cv=5,
    scoring="f1",   # 👈 important
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Best Parameters:", grid_rf.best_params_)
print("Best CV F1:", grid_rf.best_score_)

Best Parameters: {'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 10, 'rf__n_estimators': 300}
Best CV F1: 0.47295516585720765


In [64]:
best_rf = grid_rf.best_estimator_

y_pred_rf = best_rf.predict(X_test)

from sklearn.metrics import f1_score, recall_score, classification_report

print("Test F1:", f1_score(y_test, y_pred_rf))
print("Test Recall:", recall_score(y_test, y_pred_rf))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

Test F1: 0.45517241379310347
Test Recall: 0.4934579439252336

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.90      0.91      3526
           1       0.42      0.49      0.46       535

    accuracy                           0.84      4061
   macro avg       0.67      0.70      0.68      4061
weighted avg       0.86      0.84      0.85      4061



In [65]:
import numpy as np
from sklearn.metrics import f1_score

probs = best_rf.predict_proba(X_test)[:,1]

thresholds = np.arange(0.2, 0.9, 0.05)
best_f1 = 0
best_threshold = 0.5

for t in thresholds:
    y_pred_t = (probs >= t).astype(int)
    score = f1_score(y_test, y_pred_t)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best Threshold:", best_threshold)
print("Best F1 After Threshold Tuning:", best_f1)

Best Threshold: 0.44999999999999996
Best F1 After Threshold Tuning: 0.4751131221719457


In [66]:
import joblib

# Save model
joblib.dump(best_rf, "sandstorm_random_forest.pkl")

# Save threshold
joblib.dump(best_threshold, "sandstorm_threshold.pkl")

print("Model and Threshold Saved Successfully ✅")

Model and Threshold Saved Successfully ✅


import joblib
import numpy as np

# Load
loaded_model = joblib.load("sandstorm_random_forest.pkl")
loaded_threshold = joblib.load("sandstorm_threshold.pkl")

# Predict probabilities
probs = loaded_model.predict_proba(X_test)[:,1]

# Apply saved threshold
y_pred_custom = (probs >= loaded_threshold).astype(int)

Soil dryness

Wind direction

Regional pressure systems

Aerosol concentration

PM10 / PM2.5

Satellite dust index

Upper atmosphere wind